In [ ]:
# Configuração para Google Colab (instalação automática de dependências extras)
import sys
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Instalando pacotes adicionais no Google Colab...")
    # ultralytics (YOLO) e easyocr não vêm instalados por padrão
    get_ipython().system('pip install -q ultralytics easyocr pytesseract')
    get_ipython().system('apt-get install -q -y tesseract-ocr')
    print("Tudo pronto!")

# TCC Experimento 3: Pipeline Final de Vídeo (Monitoramento em Rodovias)\n\n**Objetivo**: Integrar os componentes de detecção de veículos, detecção de placas e OCR de caracteres em um pipeline unificado ponta a ponta. O sistema é projetado para processar vídeos de câmeras de monitoramento rodoviário, identificando veículos e transcrevendo suas respectivas placas em tempo real.\n\n---\n## Fluxo do Pipeline:\n1. **Leitura do Frame**: O arquivo de vídeo é aberto e processado quadro a quadro.\n2. **Detecção de Veículo**: O modelo YOLOv8 localiza veículos (`car`, `truck`, `bus`, `van`, `motorcycle`).\n3. **Recorte do Veículo (Crop)**: A região do veículo é isolada para reduzir a área de busca e diminuir falsos positivos.\n4. **Detecção de Placa**: O `PlateDetector` procura a placa de trânsito exclusivamente dentro do recorte do veículo.\n5. **Reconhecimento de Caracteres (OCR)**: A placa é isolada, pré-processada (upscaling, contraste CLAHE, threshold) e lida pelo EasyOCR.\n6. **Renderização**: Caixas delimitadoras de cores distintas e o texto decodificado são sobrepostos no frame original, que é salvo em um novo arquivo de vídeo.

## 1. Configuração e Importações

In [ ]:
import sys, cv2, time, os, warnings
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import torch

BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

warnings.filterwarnings("ignore")
print(f"CUDA disponível: {torch.cuda.is_available()}")

## 2. Inicialização dos Modelos

In [ ]:
# =============================================================================
# CÓDIGO AUXILIAR (DETECTORES E OCR) - CONSOLIDADO PARA COLAB
# =============================================================================
import torch
import cv2
import numpy as np
import time
import re
import easyocr
import pytesseract
from pathlib import Path
from abc import ABC, abstractmethod
from ultralytics import YOLO
from torchvision.models.detection import fasterrcnn_resnet50_fpn, ssdlite320_mobilenet_v3_large
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights, SSDLite320_MobileNet_V3_Large_Weights

# --- DETECTORES DE VEÍCULOS E PLACAS ---
class BaseDetector(ABC):
    @abstractmethod
    def detect(self, frame):
        """Retorna lista de detecções: [{'bbox': [x1, y1, x2, y2], 'conf': 0.9, 'class': 'car'}]"""
        pass

class YOLODetector(BaseDetector):
    def __init__(self, model_path='yolov8n.pt'):
        self.model = YOLO(model_path)
        self.model_name = "YOLOv8"
        
    def detect(self, frame):
        results = self.model(frame, verbose=False)[0]
        detections = []
        for box in results.boxes:
            detections.append({
                'bbox': box.xyxy[0].tolist(),
                'conf': float(box.conf),
                'class': self.model.names[int(box.cls)]
            })
        return detections

class TorchvisionDetector(BaseDetector):
    def __init__(self, model_type='faster_rcnn', confidence_threshold=0.5):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.threshold = confidence_threshold
        self.model_name = "Faster R-CNN" if model_type == 'faster_rcnn' else "SSD"
        
        if model_type == 'faster_rcnn':
            weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
            self.model = fasterrcnn_resnet50_fpn(weights=weights).to(self.device)
            self.classes = weights.meta["categories"]
        else: # SSD
            weights = SSDLite320_MobileNet_V3_Large_Weights.DEFAULT
            self.model = ssdlite320_mobilenet_v3_large(weights=weights).to(self.device)
            self.classes = weights.meta["categories"]
            
        self.model.eval()

    def detect(self, frame):
        # Preprocess
        img_tensor = torch.from_numpy(frame).permute(2, 0, 1).float().div(255).unsqueeze(0).to(self.device)
        
        with torch.no_grad():
            prediction = self.model(img_tensor)[0]
        
        detections = []
        for i in range(len(prediction['boxes'])):
            score = float(prediction['scores'][i])
            if score > self.threshold:
                detections.append({
                    'bbox': prediction['boxes'][i].tolist(),
                    'conf': score,
                    'class': self.classes[int(prediction['labels'][i])]
                })
        return detections

class PlateDetector(BaseDetector):
    def __init__(self, model_path='yolov8n-plate.pt'):
        try:
            # Tenta carregar o modelo YOLO especializado
            self.model = YOLO(model_path)
            self.has_model = True
        except Exception as e:
            print(f"Aviso: Modelo YOLO de placas não encontrado. Usando fallback OpenCV.")
            self.has_model = False
        self.model_name = "PlateDetector"
        
    def detect(self, frame):
        if self.has_model:
            results = self.model(frame, verbose=False)[0]
            detections = []
            for box in results.boxes:
                detections.append({
                    'bbox': box.xyxy[0].tolist(),
                    'conf': float(box.conf),
                    'class': 'plate'
                })
            return detections
        else:
            # --- FALLBACK: Visão Computacional Clássica ---
            # Ideal para TCC: Detecção baseada em bordas e contornos
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            # Filtro para reduzir ruído mantendo bordas
            bfilter = cv2.bilateralFilter(gray, 11, 17, 17)
            # Detecção de bordas
            edged = cv2.Canny(bfilter, 30, 200)
            
            # Encontrar contornos
            keypoints = cv2.findContours(edged.copy(), cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
            contours = sorted(keypoints[0], key=cv2.contourArea, reverse=True)[:10]
            
            detections = []
            for res in contours:
                approx = cv2.approxPolyDP(res, 10, True)
                if len(approx) == 4: # Retângulos (possíveis placas)
                    x, y, w, h = cv2.boundingRect(res)
                    aspect_ratio = w / float(h)
                    # Placas brasileiras têm proporção de ~3:1 a ~4:1
                    if 2.0 < aspect_ratio < 5.0:
                        detections.append({
                            'bbox': [float(x), float(y), float(x+w), float(y+h)],
                            'conf': 0.8, # Confiança fixa para o fallback
                            'class': 'plate'
                        })
            return detections

# --- OCR ENGINE ---

# ---------------------------------------------------------------------------
# Mapeamentos de correção por posição para placas brasileiras
#
# Formato antigo:  L L L N N N N   (ex: MLS5511)
# Formato Mercosul: L L L N L N N  (ex: ABC1D23)
#
# O OCR frequentemente troca:
#   letras por números: O↔0, I↔1, S↔5, B↔8, Z↔2, G↔6, Q↔0
#   números por letras: 0↔O, 1↔I, 5↔S, 8↔B, 2↔Z, 6↔G
# ---------------------------------------------------------------------------

# Caracteres que são letras mas parecem números
_NUM_TO_LETTER = str.maketrans('0158268479', 'OISBZGAHQ?')  # só se position for letra
# Caracteres que são números mas parecem letras
_LETTER_TO_NUM = str.maketrans('OISBZGAHQ', '015826649')   # só se position for dígito


def _fix_char(ch: str, expect_letter: bool) -> str:
    """Corrige um caractere OCR com base no tipo esperado na posição."""
    ch = ch.upper()
    if expect_letter:
        return ch.translate(_NUM_TO_LETTER)
    else:
        return ch.translate(_LETTER_TO_NUM)


def _is_mercosul(raw: str) -> bool:
    """Heurística para detectar se a placa está no formato Mercosul (AAA0A00)."""
    if len(raw) != 7:
        return False
    # Mercosul: posição 4 (índice 3) é número, posição 5 (índice 4) é letra
    # Antigo:   posições 4-7 (índices 3-6) são todos números
    return raw[4].isalpha() if raw[4].isascii() else False


def _apply_plate_mask(raw: str) -> str:
    """
    Aplica máscara de posição para corrigir confusões letra/número do OCR.

    Formato antigo:   L L L N N N N  (posições 0,1,2 = letra; 3,4,5,6 = dígito)
    Formato Mercosul: L L L N L N N  (posições 0,1,2 = letra; 3 = dígito;
                                       4 = letra; 5,6 = dígito)
    """
    if len(raw) < 7:
        return raw  # Muito curto — não tenta corrigir

    mercosul = _is_mercosul(raw)

    if mercosul:
        mask = [True, True, True, False, True, False, False]  # True = espera letra
    else:
        mask = [True, True, True, False, False, False, False]

    corrected = []
    for i, ch in enumerate(raw[:7]):
        if i < len(mask):
            corrected.append(_fix_char(ch, expect_letter=mask[i]))
        else:
            corrected.append(ch)

    return ''.join(corrected)


class OCREngine:
    def __init__(self, engine_type='easyocr'):
        self.engine_type = engine_type
        if engine_type == 'easyocr':
            # Português + inglês; GPU se disponível
            self.reader = easyocr.Reader(['pt', 'en'], gpu=True)

    # ------------------------------------------------------------------
    # Limpeza de texto bruto
    # ------------------------------------------------------------------
    def clean_text(self, text: str) -> str:
        """Remove tudo que não é letra ou dígito e converte para maiúsculo."""
        return re.sub(r'[^A-Z0-9]', '', text.upper())

    # ------------------------------------------------------------------
    # Pré-processamento da imagem da placa
    # ------------------------------------------------------------------
    def preprocess_plate(self, plate_crop: np.ndarray) -> np.ndarray:
        """
        Pipeline de pré-processamento otimizado para placas brasileiras.

        1. Upscaling 3× (melhora resolução para o OCR)
        2. Conversão para cinza
        3. CLAHE (equalização adaptativa de histograma — melhora contraste)
        4. Denoising leve (preserva bordas de caracteres)
        5. Threshold de Otsu (binarização global — mais estável que o adaptativo
           para placas com fundo uniforme)
        """
        h, w = plate_crop.shape[:2]
        # 1. Upscaling
        plate = cv2.resize(plate_crop, (w * 3, h * 3), interpolation=cv2.INTER_CUBIC)

        # 2. Cinza
        gray = cv2.cvtColor(plate, cv2.COLOR_BGR2GRAY)

        # 3. CLAHE — melhora contraste local sem destruir bordas finas
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
        gray = clahe.apply(gray)

        # 4. Denoising leve
        gray = cv2.fastNlMeansDenoising(gray, h=10, templateWindowSize=7, searchWindowSize=21)

        # 5. Threshold de Otsu
        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        return binary

    # ------------------------------------------------------------------
    # Leitura da placa
    # ------------------------------------------------------------------
    def read_plate(self, plate_crop: np.ndarray) -> str:
        """
        Lê o texto de uma imagem de placa recortada.

        Etapas:
          1. Pré-processamento da imagem
          2. OCR (EasyOCR ou Tesseract)
          3. Limpeza do texto bruto
          4. Correção posicional letra/número (máscara de placa brasileira)
        """
        if plate_crop is None or plate_crop.size == 0:
            return ""

        processed = self.preprocess_plate(plate_crop)

        raw = ""
        if self.engine_type == 'easyocr':
            results = self.reader.readtext(processed)
            if results:
                # Pega a leitura com maior confiança
                raw = max(results, key=lambda x: x[2])[1]
        else:
            config = (
                '--psm 7 '
                '-c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
            )
            raw = pytesseract.image_to_string(processed, config=config)

        cleaned = self.clean_text(raw)

        # Aplica correção posicional se o texto tiver comprimento de placa válido (7)
        if len(cleaned) == 7:
            cleaned = _apply_plate_mask(cleaned)

        return cleaned


In [ ]:

print("Inicializando modelos...")
# 1. Detector de veículos (usando YOLOv8n padrão COCO)
vehicle_detector = YOLODetector(str(BASE_DIR / "models" / "yolov8n.pt"))

# 2. Detector de placas (usará YOLOv8-plate ou o fallback OpenCV clássico)
plate_detector = PlateDetector()

# 3. Engine de OCR (EasyOCR baseada em Deep Learning)
ocr_engine = OCREngine(engine_type="easyocr")
print("Modelos carregados com sucesso!")

## 3. Implementação do Pipeline de Vídeo

In [ ]:
def process_highway_video(input_path, output_path, max_frames=None, vehicle_classes=None):
    if vehicle_classes is None:
        vehicle_classes = {"car", "truck", "bus", "van", "motorcycle"}

    cap = cv2.VideoCapture(str(input_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Não foi possível abrir o vídeo em: {input_path}")

    # Metadados do vídeo
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if max_frames:
        total_frames = min(total_frames, max_frames)

    # Cria o diretório de destino se não existir
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    # Configura o gravador de vídeo (codec mp4v para MP4)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))

    print(f"Processando vídeo: {input_path.name} ({width}x{height} @ {fps:.2f} FPS)")
    
    pbar = tqdm(total=total_frames, desc="Processando frames")
    frame_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or (max_frames and frame_count >= max_frames):
            break

        # 1. Detectar veículos
        detections = vehicle_detector.detect(frame)
        
        for d in detections:
            cls_name = str(d["class"]).lower()
            if cls_name not in vehicle_classes:
                continue
            
            vx1, vy1, vx2, vy2 = map(int, d["bbox"])
            # Clipa coordenadas para garantir limites da imagem
            vx1, vy1 = max(0, vx1), max(0, vy1)
            vx2, vy2 = min(width, vx2), min(height, vy2)
            
            # Desenha retângulo do Veículo (Azul)
            cv2.rectangle(frame, (vx1, vy1), (vx2, vy2), (255, 0, 0), 2)
            
            # Recorta veículo
            veh_crop = frame[vy1:vy2, vx1:vx2]
            if veh_crop.size == 0:
                continue
                
            # 2. Detectar placas dentro do crop do veículo
            plates = plate_detector.detect(veh_crop)
            
            for p in plates:
                px1, py1, px2, py2 = map(int, p["bbox"])
                
                # Coordenadas absolutas da placa no frame original
                abs_px1 = vx1 + px1
                abs_py1 = vy1 + py1
                abs_px2 = vx1 + px2
                abs_py2 = vy1 + py2
                
                # Desenha retângulo da Placa (Vermelho)
                cv2.rectangle(frame, (abs_px1, abs_py1), (abs_px2, abs_py2), (0, 0, 255), 2)
                
                # Recorta placa para OCR
                plate_crop = veh_crop[py1:py2, px1:px2]
                if plate_crop.size == 0:
                    continue
                    
                # 3. Rodar OCR
                plate_text = ocr_engine.read_plate(plate_crop)
                
                if plate_text:
                    # Desenha o texto da placa (Fundo branco com texto preto acima do veículo)
                    label = f"{cls_name.upper()}: {plate_text}"
                    (w_lbl, h_lbl), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
                    cv2.rectangle(frame, (vx1, vy1 - h_lbl - 10), (vx1 + w_lbl + 10, vy1), (255, 255, 255), -1)
                    cv2.putText(frame, label, (vx1 + 5, vy1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2, cv2.LINE_AA)

        # Grava frame anotado
        out.write(frame)
        frame_count += 1
        pbar.update(1)
        
    cap.release()
    out.release()
    pbar.close()
    print(f"Vídeo processado com sucesso e salvo em: {output_path}")

## 4. Testando o Pipeline com um Vídeo\n\nPara testar, coloque um arquivo de vídeo (ex: `video_teste.mp4`) na raiz do seu projeto ou modifique o caminho abaixo.

In [ ]:
VIDEO_ENTRADA = BASE_DIR / "video_teste.mp4"
VIDEO_SAIDA   = BASE_DIR / "data" / "processado" / "resultados" / "video_processado.mp4"

if VIDEO_ENTRADA.exists():
    # Roda para os primeiros 150 frames (~5 segundos de vídeo)
    process_highway_video(VIDEO_ENTRADA, VIDEO_SAIDA, max_frames=150)
else:
    print(f"[AVISO] Vídeo de entrada não encontrado em: {VIDEO_ENTRADA}")
    print("Por favor, insira um arquivo de vídeo com o nome \\video_teste.mp4\\ na raiz do seu projeto para realizar o teste.")

## 5. Visualizando Amostra do Frame Processado\n\nCaso tenha processado o vídeo com sucesso, você pode exibir um frame processado abaixo.

In [ ]:
if VIDEO_SAIDA.exists():
    cap = cv2.VideoCapture(str(VIDEO_SAIDA))
    ret, frame = cap.read()
    if ret:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(12, 8))
        plt.imshow(frame_rgb)
        plt.axis("off")
        plt.title("Frame Processado com Deteções (Veículo + Placa + OCR)")
        plt.show()
    cap.release()
else:
    print("Execute a célula anterior com um vídeo válido para visualizar o resultado aqui.")